# Focused Rolling Single-Ticker Experiment

Purpose: keep the rolling-window experiment separate, clear, and simple.

This notebook trains transparent benchmark models for selected tickers using a 3-year rolling training window and 1-year test window. It supports perfect-hindsight labels, RL/action labels, and long current PnL. It exports fold-level metrics and row-level predictions in a simulator-friendly format.

Default first run: `AAPL` only. Later, change `TICKERS = ['AAPL', 'PG', 'NVDA', 'BAC']`.

In [2]:
# ============================================================
# 1. Imports and global settings
# ============================================================
import os
import json
import gzip
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score, explained_variance_score,
    accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
N_JOBS = -1

# Canonical column names.
INDEX_COL = 'IndexReference'
DATE_COL = 'attr__timestamp'
TICKER_COL = 'attr__ticker'
SIC2_COL = 'attr__sic2'
YEAR_COL = 'year'
SPLIT_COL = 'split'
TRUE_COL = 'y_true'
PRED_COL = 'y_pred'
SCORE_COL = 'prediction_score'
SIGNAL_SCORE_COL = 'signal_score'
DIRECTION_COL = 'direction'
CONFIDENCE_COL = 'confidence'
RL_TRAINABLE_COL = 'label__rl.trainable'

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / 'focused_rolling_single_ticker_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT.resolve())
print('Output directory:', OUTPUT_DIR.resolve())


Project root: C:\Users\user\Downloads\universe_100_recent_post_normalisation_oneticker
Output directory: C:\Users\user\Downloads\universe_100_recent_post_normalisation_oneticker\focused_rolling_single_ticker_outputs


In [3]:
# ============================================================
# 2. User configuration
# ============================================================
# Change these paths if your files are stored elsewhere.
UNIVERSE = 'universe_100'
TIME_HORIZON = 'recent'
NORMALISATION_STATUS = 'post_normalisation'

DATASET_CONFIG = {
    'train_path': PROJECT_ROOT / f'{UNIVERSE}_{TIME_HORIZON}_{NORMALISATION_STATUS}_train.jsonl',
    'valid_path': PROJECT_ROOT / f'{UNIVERSE}_{TIME_HORIZON}_{NORMALISATION_STATUS}_validation.jsonl',
    'test_path':  PROJECT_ROOT / f'{UNIVERSE}_{TIME_HORIZON}_{NORMALISATION_STATUS}_test.jsonl',
    'project_b_feature_file': PROJECT_ROOT / 'feature_project_b.txt',
}

# Start simple.
TICKERS = ['AAPL']
# Later robustness version:
# TICKERS = ['AAPL', 'PG', 'NVDA', 'BAC']

TARGET_NAMES = [
    'pi_hindsight_entry_long',
    'pi_hindsight_entry_positive',
    'pi_hindsight_entry_original',
    'pi_hindsight_entry_6bins',
    'rl_expert_action',
    'rl_long_action_binary',
    'rl_long_is_best',
    'rl_long_action_quality',
    'rl_long_current_pnl',
]

FEATURE_SET_NAME = 'combined_project_b'
TRAIN_WINDOW_YEARS = 3
N_TEST_FOLDS = 3
TEST_YEARS = None  # Example: [2023, 2024, 2025]. If None, latest feasible 3 test years are used.
MIN_TRAIN_ROWS = 250
MIN_TEST_ROWS = 40
TOP_BOTTOM_PCT = 0.10
LONG_SIGNAL_THRESHOLD = 0.90

# Use smaller model sizes first to keep runtime manageable. Increase later if needed.
REGRESSION_MODELS = ['RandomForest', 'LightGBM']
BINARY_MODELS = ['Logistic', 'RandomForest', 'LightGBM']
MULTICLASS_MODELS = ['LogisticMultinomial', 'RandomForest', 'LightGBM']

for k, p in DATASET_CONFIG.items():
    print(k, '->', p, '| exists:', Path(p).exists())


train_path -> c:\Users\user\Downloads\universe_100_recent_post_normalisation_oneticker\universe_100_recent_post_normalisation_train.jsonl | exists: False
valid_path -> c:\Users\user\Downloads\universe_100_recent_post_normalisation_oneticker\universe_100_recent_post_normalisation_validation.jsonl | exists: False
test_path -> c:\Users\user\Downloads\universe_100_recent_post_normalisation_oneticker\universe_100_recent_post_normalisation_test.jsonl | exists: False
project_b_feature_file -> c:\Users\user\Downloads\universe_100_recent_post_normalisation_oneticker\feature_project_b.txt | exists: True


In [4]:
# ============================================================
# 3. Data loading utilities
# ============================================================
def _open_text(path):
    path = Path(path)
    if path.suffix.lower() == '.gz':
        return gzip.open(path, 'rt', encoding='utf-8')
    return path.open('r', encoding='utf-8')


def flatten_record(raw_record):
    """Flatten Adaptive Swarm JSONL rows into attr__/feature__/label__ columns."""
    if raw_record.get('section') == 'header':
        return None
    if raw_record.get('section') == 'data' and isinstance(raw_record.get('data'), dict):
        rec = raw_record['data']
    else:
        rec = raw_record
    row = {}
    if INDEX_COL in rec:
        row[INDEX_COL] = rec.get(INDEX_COL)
    for family, prefix in [('Attributes', 'attr__'), ('attributes', 'attr__'), ('Features', 'feature__'), ('features', 'feature__'), ('Labels', 'label__'), ('labels', 'label__')]:
        obj = rec.get(family)
        if isinstance(obj, dict):
            for k, v in obj.items():
                row[f'{prefix}{k}'] = v
    # Keep any already-flat columns too.
    for k, v in rec.items():
        if k not in ['Attributes', 'attributes', 'Features', 'features', 'Labels', 'labels'] and k not in row:
            row[k] = v
    return row


def load_jsonl(path):
    rows = []
    with _open_text(path) as f:
        for line in f:
            if not line.strip():
                continue
            row = flatten_record(json.loads(line))
            if row is not None:
                rows.append(row)
    return pd.DataFrame(rows)


def load_any_table(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    suffixes = ''.join(path.suffixes).lower()
    if suffixes.endswith('.jsonl') or suffixes.endswith('.jsonl.gz'):
        return load_jsonl(path)
    if suffixes.endswith('.parquet'):
        return pd.read_parquet(path)
    if suffixes.endswith('.csv'):
        return pd.read_csv(path)
    raise ValueError(f'Unsupported file type: {path}')


def load_dataset_from_config(config):
    frames = {}
    for split, key in [('train', 'train_path'), ('valid', 'valid_path'), ('test', 'test_path')]:
        path = Path(config[key])
        if path.exists():
            df = load_any_table(path)
            df[SPLIT_COL] = split
            frames[split] = df
            print(split, df.shape, path.name)
        else:
            print(f'WARNING: {split} path not found:', path)
    if not frames:
        raise FileNotFoundError('No train/valid/test files were found. Update DATASET_CONFIG paths first.')
    all_data = pd.concat(frames.values(), ignore_index=True, sort=False)
    return frames, all_data


In [5]:
# ============================================================
# 4. Target construction
# ============================================================
def first_existing_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def make_pi_hindsight_entry_long_6bins(series):
    s = pd.to_numeric(series, errors='coerce')
    out = pd.Series(np.nan, index=s.index)
    out[s == 0] = 0
    out[(s > 0) & (s < 0.1)] = 1
    out[(s >= 0.1) & (s < 0.2)] = 2
    out[(s >= 0.2) & (s < 0.3)] = 3
    out[(s >= 0.3) & (s < 0.4)] = 4
    out[s >= 0.4] = 5
    return out.astype('Int64')


def add_derived_targets(df):
    df = df.copy()

    pi_col = first_existing_column(df, [
        'label__pi_hindsight_entry_long', 'label__pi_long_entry', 'pi_hindsight_entry_long',
        'target__pi_hindsight_entry_long'
    ])
    if pi_col is not None:
        df['target__pi_hindsight_entry_long'] = pd.to_numeric(df[pi_col], errors='coerce')
        df['target__pi_hindsight_entry_positive'] = (df['target__pi_hindsight_entry_long'] > 0).astype('Int64')
        df['target__pi_hindsight_entry_original'] = (df['target__pi_hindsight_entry_long'] >= 0.4).astype('Int64')
        df['target__pi_hindsight_entry_6bins'] = make_pi_hindsight_entry_long_6bins(df['target__pi_hindsight_entry_long'])

    col = first_existing_column(df, ['label__rl.expert_action', 'label__rl_expert_action', 'target__rl_expert_action'])
    if col is not None:
        df['target__rl_expert_action'] = df[col]

    col = first_existing_column(df, ['label__rl.long.action_label', 'label__rl_long_action_binary', 'target__rl_long_action_binary'])
    if col is not None:
        df['target__rl_long_action_binary'] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

    col = first_existing_column(df, ['label__rl.long_is_best', 'label__rl_long_is_best', 'target__rl_long_is_best'])
    if col is not None:
        df['target__rl_long_is_best'] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

    col = first_existing_column(df, ['label__rl.long.action_quality', 'label__rl_long_action_quality', 'target__rl_long_action_quality'])
    if col is not None:
        df['target__rl_long_action_quality'] = pd.to_numeric(df[col], errors='coerce')

    col = first_existing_column(df, ['label__rl.long.current_pnl', 'label__rl_long_current_pnl', 'label__rl.long_current_pnl', 'label__current_pnl', 'target__rl_long_current_pnl'])
    if col is not None:
        df['target__rl_long_current_pnl'] = pd.to_numeric(df[col], errors='coerce')

    return df


TARGET_CONFIGS = {
    'pi_hindsight_entry_long': {'column': 'target__pi_hindsight_entry_long', 'task': 'regression'},
    'pi_hindsight_entry_positive': {'column': 'target__pi_hindsight_entry_positive', 'task': 'binary'},
    'pi_hindsight_entry_original': {'column': 'target__pi_hindsight_entry_original', 'task': 'binary'},
    'pi_hindsight_entry_6bins': {'column': 'target__pi_hindsight_entry_6bins', 'task': 'multiclass'},
    'rl_expert_action': {'column': 'target__rl_expert_action', 'task': 'multiclass'},
    'rl_long_action_binary': {'column': 'target__rl_long_action_binary', 'task': 'binary'},
    'rl_long_is_best': {'column': 'target__rl_long_is_best', 'task': 'binary'},
    'rl_long_action_quality': {'column': 'target__rl_long_action_quality', 'task': 'regression'},
    'rl_long_current_pnl': {'column': 'target__rl_long_current_pnl', 'task': 'regression'},
}


In [6]:
# ============================================================
# 5. Feature selection
# ============================================================
def read_project_b_features(path):
    path = Path(path)
    if not path.exists():
        return []
    feats = []
    for line in path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        feats.append(line)
    return feats


def build_feature_sets(df, config):
    numeric_feature_cols = [
        c for c in df.columns
        if c.startswith('feature__') and pd.api.types.is_numeric_dtype(pd.to_numeric(df[c], errors='coerce'))
    ]
    project_b_raw = read_project_b_features(config.get('project_b_feature_file', ''))
    # Support feature names either with or without feature__ prefix.
    project_b = []
    for f in project_b_raw:
        candidates = [f, f'feature__{f}' if not f.startswith('feature__') else f]
        for c in candidates:
            if c in df.columns:
                project_b.append(c)
                break
    project_b = sorted(set(project_b))
    if not project_b:
        project_b = numeric_feature_cols
    return {
        'combined_project_b': project_b,
        'all_numeric_features': numeric_feature_cols,
    }


In [7]:
# ============================================================
# 6. Models and metrics
# ============================================================
def make_numeric_preprocessor(scale=False):
    steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale:
        steps.append(('scaler', StandardScaler()))
    return Pipeline(steps)


def make_regression_models():
    models = {
        'DummyMean': DummyRegressor(strategy='mean'),
        'ElasticNet': make_pipeline(make_numeric_preprocessor(True), ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000, random_state=RANDOM_STATE)),
        'RandomForest': make_pipeline(make_numeric_preprocessor(False), RandomForestRegressor(n_estimators=200, min_samples_leaf=10, random_state=RANDOM_STATE, n_jobs=N_JOBS)),
        'HistGradientBoosting': make_pipeline(make_numeric_preprocessor(False), HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, random_state=RANDOM_STATE)),
    }
    try:
        from lightgbm import LGBMRegressor
        models['LightGBM'] = make_pipeline(make_numeric_preprocessor(False), LGBMRegressor(n_estimators=300, learning_rate=0.03, num_leaves=31, subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=-1))
    except Exception:
        print('LightGBM not available; using HistGradientBoosting fallback only.')
    return models


def make_binary_models():
    models = {
        'DummyMostFrequent': DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE),
        'Logistic': make_pipeline(make_numeric_preprocessor(True), LogisticRegression(class_weight='balanced', max_iter=2000, random_state=RANDOM_STATE)),
        'RandomForest': make_pipeline(make_numeric_preprocessor(False), RandomForestClassifier(n_estimators=200, min_samples_leaf=10, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=N_JOBS)),
        'HistGradientBoosting': make_pipeline(make_numeric_preprocessor(False), HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=RANDOM_STATE)),
    }
    try:
        from lightgbm import LGBMClassifier
        models['LightGBM'] = make_pipeline(make_numeric_preprocessor(False), LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=31, subsample=0.8, colsample_bytree=0.8, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=-1))
    except Exception:
        print('LightGBM not available; using HistGradientBoosting fallback only.')
    return models


def make_multiclass_models():
    models = {
        'DummyMostFrequent': DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE),
        'LogisticMultinomial': make_pipeline(make_numeric_preprocessor(True), LogisticRegression(class_weight='balanced', max_iter=3000, random_state=RANDOM_STATE)),
        'RandomForest': make_pipeline(make_numeric_preprocessor(False), RandomForestClassifier(n_estimators=200, min_samples_leaf=10, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=N_JOBS)),
        'HistGradientBoosting': make_pipeline(make_numeric_preprocessor(False), HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=RANDOM_STATE)),
    }
    try:
        from lightgbm import LGBMClassifier
        models['LightGBM'] = make_pipeline(make_numeric_preprocessor(False), LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=31, subsample=0.8, colsample_bytree=0.8, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=-1))
    except Exception:
        print('LightGBM not available; using HistGradientBoosting fallback only.')
    return models


def get_models_for_task(task):
    if task == 'regression':
        return make_regression_models()
    if task == 'binary':
        return make_binary_models()
    if task == 'multiclass':
        return make_multiclass_models()
    raise ValueError(task)


def safe_spearman(y_true, y_score):
    y_true = pd.Series(y_true)
    y_score = pd.Series(y_score)
    valid = y_true.notna() & y_score.notna()
    if valid.sum() < 3 or y_true.loc[valid].nunique() < 2 or y_score.loc[valid].nunique() < 2:
        return np.nan
    return y_true.loc[valid].corr(y_score.loc[valid], method='spearman')


def safe_pearson(y_true, y_score):
    y_true = pd.Series(y_true)
    y_score = pd.Series(y_score)
    valid = y_true.notna() & y_score.notna()
    if valid.sum() < 3 or y_true.loc[valid].nunique() < 2 or y_score.loc[valid].nunique() < 2:
        return np.nan
    return y_true.loc[valid].corr(y_score.loc[valid], method='pearson')


def regression_metrics(y_true, y_pred):
    y_true = pd.Series(y_true).astype(float)
    y_pred = pd.Series(y_pred).astype(float)
    valid = y_true.notna() & y_pred.notna()
    if valid.sum() == 0:
        return {}
    yt = y_true.loc[valid]
    yp = y_pred.loc[valid]
    return {
        'n': int(valid.sum()),
        'mae': mean_absolute_error(yt, yp),
        'rmse': np.sqrt(mean_squared_error(yt, yp)),
        'r2': r2_score(yt, yp) if yt.nunique() > 1 else np.nan,
        'explained_variance': explained_variance_score(yt, yp) if yt.nunique() > 1 else np.nan,
        'pearson': safe_pearson(yt, yp),
        'spearman': safe_spearman(yt, yp),
        'directional_accuracy': (np.sign(yt) == np.sign(yp)).mean() if yt.nunique() > 1 and yp.nunique() > 1 else np.nan,
    }


def binary_metrics(y_true, y_pred, y_score=None):
    yt = pd.Series(y_true).dropna().astype(int)
    yp = pd.Series(y_pred).loc[yt.index].astype(int)
    out = {
        'n': int(len(yt)),
        'accuracy': accuracy_score(yt, yp),
        'balanced_accuracy': balanced_accuracy_score(yt, yp) if yt.nunique() > 1 else np.nan,
        'precision': precision_score(yt, yp, zero_division=0),
        'recall': recall_score(yt, yp, zero_division=0),
        'f1': f1_score(yt, yp, zero_division=0),
    }
    if y_score is not None:
        ys = pd.Series(y_score).loc[yt.index]
        if yt.nunique() > 1 and ys.nunique() > 1:
            out['roc_auc'] = roc_auc_score(yt, ys)
            out['pr_auc'] = average_precision_score(yt, ys)
        else:
            out['roc_auc'] = np.nan
            out['pr_auc'] = np.nan
    return out


def multiclass_metrics(y_true, y_pred):
    # Convert to string to avoid sklearn errors with pandas nullable integer / mixed object labels.
    yt = pd.Series(y_true).dropna()
    yp = pd.Series(y_pred).loc[yt.index]
    yt_eval = yt.astype(str)
    yp_eval = yp.astype(str)
    return {
        'n': int(len(yt_eval)),
        'accuracy': accuracy_score(yt_eval, yp_eval),
        'balanced_accuracy': balanced_accuracy_score(yt_eval, yp_eval) if yt_eval.nunique() > 1 else np.nan,
        'macro_f1': f1_score(yt_eval, yp_eval, average='macro', zero_division=0),
        'weighted_f1': f1_score(yt_eval, yp_eval, average='weighted', zero_division=0),
        'n_classes_true': yt_eval.nunique(),
        'n_classes_pred': yp_eval.nunique(),
    }


In [8]:
# ============================================================
# 7. Rolling helper functions
# ============================================================
def prepare_xy(df, feature_cols, target_col, task):
    data = df[[c for c in feature_cols if c in df.columns] + [target_col]].copy()
    data = data.dropna(subset=[target_col])
    X = data[[c for c in feature_cols if c in data.columns]].apply(pd.to_numeric, errors='coerce')
    y = data[target_col]
    if task == 'regression':
        y = pd.to_numeric(y, errors='coerce')
    valid = y.notna()
    return X.loc[valid], y.loc[valid]


def is_rl_target(target_name):
    return str(target_name).startswith('rl_')


def filter_rows_for_target(df, target_name):
    df = df.copy()
    if is_rl_target(target_name) and RL_TRAINABLE_COL in df.columns:
        return df[pd.to_numeric(df[RL_TRAINABLE_COL], errors='coerce') == 1].copy()
    return df


def ensure_year_column(df):
    df = df.copy()
    if DATE_COL in df.columns:
        df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce')
    if YEAR_COL not in df.columns or df[YEAR_COL].isna().all():
        if DATE_COL not in df.columns:
            raise ValueError(f'Missing both {YEAR_COL} and {DATE_COL}.')
        df[YEAR_COL] = df[DATE_COL].dt.year
    df[YEAR_COL] = pd.to_numeric(df[YEAR_COL], errors='coerce').astype('Int64')
    return df


def build_single_ticker_rolling_folds(df, ticker, train_window_years=3, n_test_folds=3, test_years=None):
    one = df[df[TICKER_COL].astype(str) == str(ticker)].copy()
    years = sorted(one[YEAR_COL].dropna().astype(int).unique().tolist())
    if test_years is None:
        feasible = []
        for y in years:
            train_years = list(range(y - train_window_years, y))
            if all(ty in years for ty in train_years):
                feasible.append(y)
        test_years = feasible[-n_test_folds:]
    else:
        test_years = [int(y) for y in test_years]
    folds = []
    for test_year in test_years:
        train_years = list(range(test_year - train_window_years, test_year))
        folds.append({
            'ticker': ticker,
            'walkforward_scheme': f'rolling_{train_window_years}y',
            'window_label': f'{min(train_years)}-{max(train_years)}_to_{test_year}',
            'fold_id': f'{ticker}_rolling_{min(train_years)}_{max(train_years)}_test_{test_year}',
            'train_years': train_years,
            'train_start_year': min(train_years),
            'train_end_year': max(train_years),
            'test_year': test_year,
        })
    return folds


def model_classes(model):
    if hasattr(model, 'classes_'):
        return list(model.classes_)
    if hasattr(model, 'named_steps'):
        last = list(model.named_steps.values())[-1]
        if hasattr(last, 'classes_'):
            return list(last.classes_)
    return None


def score_for_ranking(model, X, y_pred, task, target_name):
    if task == 'regression':
        return np.asarray(y_pred, dtype=float)
    if not hasattr(model, 'predict_proba'):
        return np.asarray(y_pred)
    proba = model.predict_proba(X)
    classes = model_classes(model)
    if classes is None:
        return proba[:, -1] if proba.ndim == 2 else proba
    classes_str = [str(c).lower() for c in classes]
    if 'long' in classes_str:
        return proba[:, classes_str.index('long')]
    if '1' in classes_str:
        return proba[:, classes_str.index('1')]
    if 1 in classes:
        return proba[:, classes.index(1)]
    numeric = pd.to_numeric(pd.Series(classes), errors='coerce')
    if numeric.notna().all():
        return np.dot(proba, numeric.to_numpy(dtype=float))
    return np.max(proba, axis=1)


def actual_value_for_ranking(y, target_name):
    s = pd.Series(y).copy()
    if target_name == 'rl_expert_action':
        return s.astype(str).str.lower().eq('long').astype(float)
    return pd.to_numeric(s, errors='coerce').astype(float)


def percentile_against_train_distribution(test_scores, train_scores):
    train_scores = pd.to_numeric(pd.Series(train_scores), errors='coerce').dropna().sort_values().to_numpy()
    test_scores = pd.to_numeric(pd.Series(test_scores), errors='coerce').to_numpy()
    if len(train_scores) == 0:
        return pd.Series(np.nan, index=range(len(test_scores)))
    pct = np.searchsorted(train_scores, test_scores, side='right') / len(train_scores)
    return pd.Series(np.clip(pct, 0, 1))


def top_bottom_diagnostics(y_true, score, pct=0.10, target_name=None):
    y_rank = actual_value_for_ranking(y_true, target_name)
    score = pd.to_numeric(pd.Series(score), errors='coerce')
    d = pd.DataFrame({'y_rank': y_rank, 'score': score}).dropna()
    if len(d) < 10 or d['score'].nunique() < 2:
        return {'top_n': 0, 'bottom_n': 0, 'top_mean_true_rank_value': np.nan, 'bottom_mean_true_rank_value': np.nan, 'top_minus_bottom_mean_true': np.nan, 'top_positive_rate': np.nan, 'bottom_positive_rate': np.nan, 'top_bottom_lift': np.nan}
    n = max(1, int(np.ceil(len(d) * pct)))
    ranked = d.sort_values('score', ascending=False)
    top = ranked.head(n)
    bottom = ranked.tail(n)
    top_pos = (top['y_rank'] > 0).mean()
    bottom_pos = (bottom['y_rank'] > 0).mean()
    return {
        'top_n': int(len(top)),
        'bottom_n': int(len(bottom)),
        'top_mean_score': top['score'].mean(),
        'bottom_mean_score': bottom['score'].mean(),
        'top_mean_true_rank_value': top['y_rank'].mean(),
        'bottom_mean_true_rank_value': bottom['y_rank'].mean(),
        'top_minus_bottom_mean_true': top['y_rank'].mean() - bottom['y_rank'].mean(),
        'top_positive_rate': top_pos,
        'bottom_positive_rate': bottom_pos,
        'top_bottom_lift': top_pos / bottom_pos if bottom_pos and bottom_pos > 0 else np.nan,
    }


In [9]:
# ============================================================
# 8. Fit one fold and calculate metrics
# ============================================================
def prediction_frame(test_df, y_test, y_pred, score, signal_score, fold, target_name, task, feature_set, model_name):
    meta_cols = [INDEX_COL, DATE_COL, YEAR_COL, TICKER_COL, SIC2_COL, SPLIT_COL]
    meta = test_df.loc[y_test.index, [c for c in meta_cols if c in test_df.columns]].copy()
    out = meta.copy()
    out['ticker'] = fold['ticker']
    out['target_name'] = target_name
    out['task'] = task
    out['feature_set'] = feature_set
    out['model_name'] = model_name
    out['model'] = model_name
    out['walkforward_scheme'] = fold['walkforward_scheme']
    out['window_label'] = fold['window_label']
    out['fold_id'] = fold['fold_id']
    out['train_start_year'] = fold['train_start_year']
    out['train_end_year'] = fold['train_end_year']
    out['test_year'] = fold['test_year']
    out[TRUE_COL] = np.asarray(y_test)
    out[PRED_COL] = np.asarray(y_pred)
    out[SCORE_COL] = np.asarray(score)
    out[SIGNAL_SCORE_COL] = np.asarray(signal_score)
    out[DIRECTION_COL] = np.where(out[SIGNAL_SCORE_COL] >= LONG_SIGNAL_THRESHOLD, 'long', 'no_trade')
    out[CONFIDENCE_COL] = out[SIGNAL_SCORE_COL]
    out['actual_value_for_ranking'] = actual_value_for_ranking(out[TRUE_COL], target_name).to_numpy()
    return out


def metrics_for_prediction_frame(pred, target_name, task):
    out = {'n': len(pred), 'target_name': target_name, 'task': task}
    y_true = pred[TRUE_COL]
    y_pred = pred[PRED_COL]
    score = pred[SCORE_COL]
    y_rank = pred['actual_value_for_ranking']
    if task == 'regression':
        out.update(regression_metrics(y_true, y_pred))
    elif task == 'binary':
        out.update(binary_metrics(y_true, y_pred, score))
    elif task == 'multiclass':
        out.update(multiclass_metrics(y_true, y_pred))
    out['within_ticker_spearman'] = safe_spearman(y_rank, score)
    out['within_ticker_pearson'] = safe_pearson(y_rank, score)
    out.update(top_bottom_diagnostics(y_true, score, TOP_BOTTOM_PCT, target_name))
    return out


def fit_predict_one_fold(base, fold, target_name, target_cfg, feature_cols, model_name, model):
    target_col = target_cfg['column']
    task = target_cfg['task']
    ticker_df = base[base[TICKER_COL].astype(str) == str(fold['ticker'])].copy()
    train_df = ticker_df[ticker_df[YEAR_COL].astype(int).isin(fold['train_years'])].copy()
    test_df = ticker_df[ticker_df[YEAR_COL].astype(int) == int(fold['test_year'])].copy()
    train_df = filter_rows_for_target(train_df, target_name)
    test_df = filter_rows_for_target(test_df, target_name)
    X_train, y_train = prepare_xy(train_df, feature_cols, target_col, task)
    X_test, y_test = prepare_xy(test_df, feature_cols, target_col, task)
    if len(y_train) < MIN_TRAIN_ROWS or len(y_test) < MIN_TEST_ROWS:
        return None, None, {'status': 'skipped', 'reason': 'too_few_rows', 'n_train': len(y_train), 'n_test': len(y_test), **fold}
    if task in ['binary', 'multiclass'] and y_train.nunique() < 2:
        return None, None, {'status': 'skipped', 'reason': 'only_one_train_class', 'n_train': len(y_train), 'n_test': len(y_test), **fold}

    fitted = clone(model)
    fitted.fit(X_train, y_train)
    y_pred = fitted.predict(X_test)
    test_score = score_for_ranking(fitted, X_test, y_pred, task, target_name)
    train_pred = fitted.predict(X_train)
    train_score = score_for_ranking(fitted, X_train, train_pred, task, target_name)
    signal_score = percentile_against_train_distribution(test_score, train_score)

    pred = prediction_frame(test_df, y_test, y_pred, test_score, signal_score, fold, target_name, task, FEATURE_SET_NAME, model_name)
    metrics = metrics_for_prediction_frame(pred, target_name, task)
    metrics.update({
        'ticker': fold['ticker'],
        'feature_set': FEATURE_SET_NAME,
        'model': model_name,
        'walkforward_scheme': fold['walkforward_scheme'],
        'window_label': fold['window_label'],
        'fold_id': fold['fold_id'],
        'train_start_year': fold['train_start_year'],
        'train_end_year': fold['train_end_year'],
        'test_year': fold['test_year'],
        'n_train': len(y_train),
        'n_test': len(y_test),
    })
    return metrics, pred, {'status': 'completed', 'n_train': len(y_train), 'n_test': len(y_test), **fold}


In [10]:
# ============================================================
# 9. Main runner
# ============================================================
def run_focused_rolling(base, feature_sets):
    base = ensure_year_column(base)
    available_targets = {k: v for k, v in TARGET_CONFIGS.items() if v['column'] in base.columns and base[v['column']].notna().any()}
    print('Available targets:', list(available_targets.keys()))
    if FEATURE_SET_NAME not in feature_sets:
        raise ValueError(f'Missing feature set {FEATURE_SET_NAME}. Available: {list(feature_sets)}')
    feature_cols = [c for c in feature_sets[FEATURE_SET_NAME] if c in base.columns]
    if not feature_cols:
        raise ValueError('No feature columns available for selected feature set.')
    print('Feature set:', FEATURE_SET_NAME, '| n_features:', len(feature_cols))

    model_name_by_task = {
        'regression': REGRESSION_MODELS,
        'binary': BINARY_MODELS,
        'multiclass': MULTICLASS_MODELS,
    }

    fold_rows, metric_rows, pred_parts, skipped_rows = [], [], [], []
    for ticker in TICKERS:
        folds = build_single_ticker_rolling_folds(base, ticker, TRAIN_WINDOW_YEARS, N_TEST_FOLDS, TEST_YEARS)
        fold_rows.extend(folds)
        if not folds:
            skipped_rows.append({'ticker': ticker, 'status': 'skipped', 'reason': 'no_feasible_folds'})
            continue
        for target_name in TARGET_NAMES:
            if target_name not in available_targets:
                skipped_rows.append({'ticker': ticker, 'target_name': target_name, 'status': 'skipped', 'reason': 'target_unavailable'})
                continue
            target_cfg = available_targets[target_name]
            models = get_models_for_task(target_cfg['task'])
            for model_name in model_name_by_task[target_cfg['task']]:
                if model_name not in models:
                    skipped_rows.append({'ticker': ticker, 'target_name': target_name, 'model': model_name, 'status': 'skipped', 'reason': 'model_unavailable'})
                    continue
                for fold in folds:
                    print(f"{ticker} | {target_name} | {model_name} | train {fold['train_start_year']}-{fold['train_end_year']} -> test {fold['test_year']}")
                    try:
                        metrics, pred, status = fit_predict_one_fold(base, fold, target_name, target_cfg, feature_cols, model_name, models[model_name])
                        if metrics is not None:
                            metric_rows.append(metrics)
                        if pred is not None and len(pred):
                            pred_parts.append(pred)
                        if status.get('status') != 'completed':
                            status.update({'target_name': target_name, 'model': model_name})
                            skipped_rows.append(status)
                    except Exception as e:
                        skipped_rows.append({'ticker': ticker, 'target_name': target_name, 'model': model_name, 'fold_id': fold['fold_id'], 'status': 'error', 'reason': str(e)})
                        print('  ERROR:', e)

    folds_df = pd.DataFrame(fold_rows)
    metrics_by_fold = pd.DataFrame(metric_rows)
    predictions = pd.concat(pred_parts, ignore_index=True, sort=False) if pred_parts else pd.DataFrame()
    skipped = pd.DataFrame(skipped_rows)

    overall_rows = []
    if len(predictions):
        group_cols = ['ticker', 'target_name', 'task', 'feature_set', 'model', 'walkforward_scheme']
        for keys, g in predictions.groupby(group_cols, dropna=False):
            target_name = keys[1]
            task = keys[2]
            row = metrics_for_prediction_frame(g, target_name, task)
            for col, value in zip(group_cols, keys):
                row[col] = value
            row['n_folds'] = g['fold_id'].nunique()
            row['test_years'] = ','.join(map(str, sorted(g['test_year'].dropna().astype(int).unique())))
            overall_rows.append(row)
    metrics_overall = pd.DataFrame(overall_rows)

    return metrics_by_fold, metrics_overall, predictions, skipped, folds_df


In [12]:
# ============================================================
# 10. Load data and run experiment
# ============================================================
frames, all_data = load_dataset_from_config(DATASET_CONFIG)
frames = {split: add_derived_targets(df) for split, df in frames.items()}
all_data = pd.concat(frames.values(), ignore_index=True, sort=False)
all_data = ensure_year_column(all_data)
FEATURE_SETS = build_feature_sets(all_data, DATASET_CONFIG)

metrics_by_fold, metrics_overall, row_level_predictions, skipped, folds = run_focused_rolling(all_data, FEATURE_SETS)

print('metrics_by_fold:', metrics_by_fold.shape)
print('metrics_overall:', metrics_overall.shape)
print('row_level_predictions:', row_level_predictions.shape)
print('skipped:', skipped.shape)


train (98419, 750) universe_100_recent_post_normalisation_train.jsonl
valid (24469, 750) universe_100_recent_post_normalisation_validation.jsonl
test (32142, 750) universe_100_recent_post_normalisation_test.jsonl
Available targets: ['pi_hindsight_entry_long', 'pi_hindsight_entry_positive', 'pi_hindsight_entry_original', 'pi_hindsight_entry_6bins', 'rl_expert_action', 'rl_long_action_binary', 'rl_long_action_quality', 'rl_long_current_pnl']
Feature set: combined_project_b | n_features: 264
AAPL | pi_hindsight_entry_long | RandomForest | train 2021-2023 -> test 2024
AAPL | pi_hindsight_entry_long | RandomForest | train 2022-2024 -> test 2025
AAPL | pi_hindsight_entry_long | RandomForest | train 2023-2025 -> test 2026
AAPL | pi_hindsight_entry_long | LightGBM | train 2021-2023 -> test 2024
AAPL | pi_hindsight_entry_long | LightGBM | train 2022-2024 -> test 2025
AAPL | pi_hindsight_entry_long | LightGBM | train 2023-2025 -> test 2026
AAPL | pi_hindsight_entry_positive | Logistic | train 20

In [14]:
# =========================
# Save outputs
# =========================

out_dir = PROJECT_ROOT / "focused_rolling_single_ticker_outputs"
out_dir.mkdir(parents=True, exist_ok=True)

metrics_by_fold_path = out_dir / "focused_rolling_metrics_by_fold.csv"
metrics_overall_path = out_dir / "focused_rolling_metrics_overall.csv"
row_predictions_path = out_dir / "focused_rolling_row_level_predictions.csv"
row_predictions_parquet_path = out_dir / "focused_rolling_row_level_predictions.parquet"
skipped_path = out_dir / "focused_rolling_skipped.csv"
excel_path = out_dir / "focused_rolling_single_ticker_results.xlsx"

metrics_by_fold.to_csv(metrics_by_fold_path, index=False)
metrics_overall.to_csv(metrics_overall_path, index=False)
row_level_predictions.to_csv(row_predictions_path, index=False)
row_level_predictions.to_parquet(row_predictions_parquet_path, index=False)
skipped.to_csv(skipped_path, index=False)

# Excel cannot store timezone-aware datetimes, so create Excel-safe copies
def make_excel_safe(df):
    df = df.copy()
    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            # Remove timezone if present
            try:
                df[col] = df[col].dt.tz_localize(None)
            except TypeError:
                # Already timezone-naive
                pass
        elif df[col].dtype == "object":
            # Handle object columns that may contain timezone-aware timestamps
            if col.lower() in ["timestamp", "date", "datetime"]:
                converted = pd.to_datetime(df[col], errors="ignore")
                if pd.api.types.is_datetime64_any_dtype(converted):
                    try:
                        converted = converted.dt.tz_localize(None)
                    except TypeError:
                        pass
                    df[col] = converted
    return df

metrics_by_fold_xlsx = make_excel_safe(metrics_by_fold)
metrics_overall_xlsx = make_excel_safe(metrics_overall)
skipped_xlsx = make_excel_safe(skipped)
prediction_preview_xlsx = make_excel_safe(row_level_predictions.head(50000))

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    metrics_by_fold_xlsx.to_excel(writer, sheet_name="metrics_by_fold", index=False)
    metrics_overall_xlsx.to_excel(writer, sheet_name="metrics_overall", index=False)
    skipped_xlsx.to_excel(writer, sheet_name="skipped", index=False)
    prediction_preview_xlsx.to_excel(writer, sheet_name="prediction_preview", index=False)

print("Saved to:", out_dir.resolve())
print("Excel:", excel_path.resolve())
print("Row-level predictions CSV:", row_predictions_path.resolve())
print("Row-level predictions parquet:", row_predictions_parquet_path.resolve())

Saved to: C:\Users\user\Downloads\universe_100_recent_post_normalisation_oneticker\focused_rolling_single_ticker_outputs
Excel: C:\Users\user\Downloads\universe_100_recent_post_normalisation_oneticker\focused_rolling_single_ticker_outputs\focused_rolling_single_ticker_results.xlsx
Row-level predictions CSV: C:\Users\user\Downloads\universe_100_recent_post_normalisation_oneticker\focused_rolling_single_ticker_outputs\focused_rolling_row_level_predictions.csv
Row-level predictions parquet: C:\Users\user\Downloads\universe_100_recent_post_normalisation_oneticker\focused_rolling_single_ticker_outputs\focused_rolling_row_level_predictions.parquet


## Export row-level predictions to strict prediction JSON

This section converts the saved row-level prediction parquet into the minimal prediction submission JSON structure discussed for long-only simulator-compatible benchmarking. The exported JSON keeps only the required submission structure plus the selected model output fields. Additional rule details should be documented in a separate report.

Current export rule:

- target: `rl_long_current_pnl` → `rl.long.current_pnl`
- models: `RandomForest`, `LightGBM`
- prediction: `signal_score`
- confidence: `1.0`
- direction: `long` if `signal_score >= 0.90`, otherwise `no_trade`
- low long-side scores are **not** converted into short signals


In [ ]:
# =========================
# Export parquet predictions to strict prediction JSON
# =========================

from pathlib import Path
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd

# -------------------------
# Export configuration
# -------------------------

# Uses the parquet generated by the previous "Save outputs" cell.
# If you saved outputs to a ticker-specific folder, change this path accordingly.
PREDICTION_PARQUET_PATH = out_dir / "focused_rolling_row_level_predictions.parquet"

PREDICTION_JSON_OUTPUT_DIR = out_dir / "prediction_json_exports"
PREDICTION_JSON_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPORT_TARGET_NAME = "rl_long_current_pnl"
EXPORT_MODEL_NAMES = ["RandomForest", "LightGBM"]

SIGNAL_THRESHOLD = 0.90
CONFIDENCE_VALUE = 1.0

PREDICTION_SCHEMA_VERSION = "1.0"
EXPERIMENT_ID = "FOCUSED-ROLLING-SINGLE-TICKER"
SPLIT_NAME = "test"
TIMEFRAME_VALUE = "d"

INCLUDE_AUDIT_VALUES = False
# Set INCLUDE_AUDIT_VALUES = True only for internal checking.
# For final blind/test submission, keep it False because actual outcomes should usually be omitted.


# -------------------------
# Mapping helpers
# -------------------------

MODEL_TYPE_MAP = {
    "RandomForest": "random_forest",
    "LightGBM": "lightgbm",
    "Logistic": "logistic_regression",
    "LogisticMultinomial": "logistic_regression",
}

TARGET_LABEL_MAP = {
    "pi_hindsight_entry_long": "pi_hindsight_entry_long",
    "pi_hindsight_entry_positive": "pi_hindsight_entry_long.positive",
    "pi_hindsight_entry_original": "pi_hindsight_entry_long.original_entry_binary",
    "pi_hindsight_entry_6bins": "pi_hindsight_entry_long.6bins",
    "rl_expert_action": "rl.expert_action",
    "rl_long_action_binary": "rl.long.action_label",
    "rl_long_action_quality": "rl.long.action_quality",
    "rl_long_current_pnl": "rl.long.current_pnl",
}


def model_name_to_model_type(model_name: str) -> str:
    """Map notebook model name to the desired submission model_type."""
    if model_name in MODEL_TYPE_MAP:
        return MODEL_TYPE_MAP[model_name]

    model_name_lower = str(model_name).lower()
    if "random" in model_name_lower and "forest" in model_name_lower:
        return "random_forest"
    if "lightgbm" in model_name_lower or "lgbm" in model_name_lower:
        return "lightgbm"
    if "logistic" in model_name_lower:
        return "logistic_regression"
    return model_name_lower.replace(" ", "_")


def target_name_to_target_label(target_name: str) -> str:
    """Map notebook target name to the dataset/company-style label."""
    return TARGET_LABEL_MAP.get(target_name, target_name)


def to_json_safe(value):
    """Convert pandas/numpy values into JSON-safe Python values."""
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except TypeError:
        pass

    if isinstance(value, pd.Timestamp):
        if value.tzinfo is None:
            value = value.tz_localize("UTC")
        else:
            value = value.tz_convert("UTC")
        return value.isoformat().replace("+00:00", "Z")

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    if isinstance(value, np.bool_):
        return bool(value)

    return value


def get_timestamp_value(row: pd.Series) -> str:
    """Return timestamp in ISO UTC format for submission audit field."""
    timestamp_col = "attr__timestamp" if "attr__timestamp" in row.index else "timestamp"
    ts = pd.to_datetime(row[timestamp_col], errors="coerce")
    if pd.isna(ts):
        raise ValueError(f"Invalid timestamp for IndexReference={row.get('IndexReference')}")
    return to_json_safe(ts)


def build_prediction_result(row: pd.Series, target_label: str) -> dict:
    """Build one strict prediction_results item."""
    signal_score = float(row["signal_score"])
    raw_prediction = float(row["prediction_score"])
    direction = "long" if signal_score >= SIGNAL_THRESHOLD else "no_trade"

    prediction_item = {
        "index": int(row["IndexReference"]),
        "ticker": str(row["ticker"]),
        "timestamp": get_timestamp_value(row),
        "timeframe": TIMEFRAME_VALUE,
        "predictions": {
            "trade.long": {
                "label": target_label,
                "prediction": signal_score,
                "confidence": CONFIDENCE_VALUE,
                "raw_prediction": raw_prediction,
                "signal_score": signal_score,
                "direction": direction,
            }
        },
    }

    if INCLUDE_AUDIT_VALUES:
        # Internal checking only. Usually omit actual outcomes from final blind/test submissions.
        if "y_true" in row.index:
            prediction_item["predictions"]["trade.long"]["actual_value"] = to_json_safe(row["y_true"])
        if "actual_value_for_ranking" in row.index:
            prediction_item["predictions"]["trade.long"]["actual_value_for_ranking"] = to_json_safe(
                row["actual_value_for_ranking"]
            )
        if "fold_id" in row.index:
            prediction_item["fold_id"] = to_json_safe(row["fold_id"])
        if "test_year" in row.index:
            prediction_item["test_year"] = to_json_safe(row["test_year"])

    return prediction_item


def build_prediction_submission(df_one: pd.DataFrame, model_name: str, target_name: str) -> dict:
    """Build one single-JSON prediction submission object."""
    target_label = target_name_to_target_label(target_name)
    model_type = model_name_to_model_type(model_name)

    tickers = sorted(df_one["ticker"].dropna().astype(str).unique().tolist())
    ticker_label = tickers[0] if len(tickers) == 1 else "MULTI"

    model_id = f"{ticker_label}_{model_name}_{target_name}_long_only_top10"

    metadata = {
        "prediction_schema_version": PREDICTION_SCHEMA_VERSION,
        "experiment_id": EXPERIMENT_ID,
        "split": SPLIT_NAME,
        "model_id": model_id,
        "model_type": model_type,
        "target_label": target_label,
        "created_at": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
        "notes": (
            "Long-only benchmark prediction file. Prediction rule, signal_score definition, "
            "confidence policy, and rolling-window design are documented separately."
        ),
    }

    prediction_results = [
        build_prediction_result(row, target_label=target_label)
        for _, row in df_one.iterrows()
    ]

    return {
        "metadata": metadata,
        "prediction_results": prediction_results,
    }


def export_prediction_json(df: pd.DataFrame, model_name: str, target_name: str) -> Path | None:
    """Filter one model-target pair and export one strict JSON file."""
    df_one = df[
        (df["target_name"] == target_name)
        & (df["model_name"] == model_name)
    ].copy()

    if df_one.empty:
        print(f"Skipped: no rows for model={model_name}, target={target_name}")
        return None

    sort_cols = [col for col in ["ticker", "attr__timestamp", "fold_id", "IndexReference"] if col in df_one.columns]
    if sort_cols:
        df_one = df_one.sort_values(sort_cols).reset_index(drop=True)

    submission = build_prediction_submission(df_one, model_name=model_name, target_name=target_name)

    ticker_label = submission["metadata"]["model_id"].split("_")[0]
    output_path = PREDICTION_JSON_OUTPUT_DIR / f"{submission['metadata']['model_id']}_predictions.json"

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(submission, f, ensure_ascii=False, indent=2)

    long_count = sum(
        item["predictions"]["trade.long"]["direction"] == "long"
        for item in submission["prediction_results"]
    )
    total_count = len(submission["prediction_results"])
    long_rate = long_count / total_count if total_count else 0.0

    print(f"Exported: {output_path}")
    print(f"  model_type: {submission['metadata']['model_type']}")
    print(f"  target_label: {submission['metadata']['target_label']}")
    print(f"  rows: {total_count}")
    print(f"  long rows: {long_count} ({long_rate:.2%})")

    return output_path


# -------------------------
# Load, validate, and export
# -------------------------

row_predictions_for_export = pd.read_parquet(PREDICTION_PARQUET_PATH)

required_columns = [
    "IndexReference",
    "ticker",
    "target_name",
    "model_name",
    "prediction_score",
    "signal_score",
]
missing_columns = [col for col in required_columns if col not in row_predictions_for_export.columns]
if missing_columns:
    raise ValueError(f"Missing required columns in parquet: {missing_columns}")

if "attr__timestamp" not in row_predictions_for_export.columns and "timestamp" not in row_predictions_for_export.columns:
    raise ValueError("Missing timestamp column. Expected either 'attr__timestamp' or 'timestamp'.")

print("Loaded prediction parquet")
print("Rows:", len(row_predictions_for_export))
print("Targets:", sorted(row_predictions_for_export["target_name"].dropna().unique().tolist()))
print("Models:", sorted(row_predictions_for_export["model_name"].dropna().unique().tolist()))
print("Tickers:", sorted(row_predictions_for_export["ticker"].dropna().astype(str).unique().tolist()))
print()

exported_prediction_json_files = []
for model_name in EXPORT_MODEL_NAMES:
    path = export_prediction_json(
        row_predictions_for_export,
        model_name=model_name,
        target_name=EXPORT_TARGET_NAME,
    )
    if path is not None:
        exported_prediction_json_files.append(path)

print("\nDone. Exported prediction JSON files:")
for path in exported_prediction_json_files:
    print("-", path)
